# 06 · Demo: change the background, watch the prediction
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mundmina/dl_project/blob/main/notebooks/06_demo.ipynb)

**Runtime → T4 GPU → Run all**, затем открой ссылку `https://….gradio.live` из последней ячейки
(работает 72 часа, можно открыть на защите с любого компьютера).

Загружаешь фото кошки или собаки → сеть сегментации (DeepLabV3) вырезает животное → выбираешь новый фон →
модели A и B (ResNet-18, seed 0) показывают топ-3 пород для оригинала и для нового фона.

In [ ]:
# Shared setup: Drive, repo, dependencies. Same cell in every notebook.
# Colab: nothing to set. Local machine (own GPU): see LOCAL.md — open this notebook from the repo's notebooks/
# folder and set BGBIAS_DRIVE, BGBIAS_DATA and GITHUB_TOKEN before starting Jupyter.
REPO_URL = "https://github.com/mundmina/dl_project.git"
BRANCH = "main"

import os, subprocess, sys
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_DIR = "/content/dl_project"
    drive.mount("/content/drive")
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
    else:  # always run the latest code from GitHub
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"], check=True)
else:
    REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))   # notebook opened from <repo>/notebooks
    assert os.path.exists(os.path.join(REPO_DIR, "src", "bgbias")), "open the notebook from the repo's notebooks/ folder"
    missing = [v for v in ("BGBIAS_DRIVE", "BGBIAS_DATA") if not os.environ.get(v)]
    assert not missing, f"set {missing} before starting Jupyter (see LOCAL.md)"
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], check=True)
commit = subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout
print("code version:", commit)
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")
# DataLoader workers on Windows start fresh interpreters: they find bgbias through PYTHONPATH, not sys.path
os.environ["PYTHONPATH"] = os.pathsep.join(p for p in (f"{REPO_DIR}/src", os.environ.get("PYTHONPATH")) if p)
try:  # ask for secret access now, not at the end of a long run (the prompt times out unattended)
    if not os.environ.get("GITHUB_TOKEN"):
        from google.colab import userdata
        userdata.get("GITHUB_TOKEN")
    print("GITHUB_TOKEN: ok")
except Exception as e:  # noqa: BLE001
    print(f"GITHUB_TOKEN not available ({type(e).__name__}): results will not be pushed to GitHub")

In [ ]:
import torch
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"], check=True)

In [ ]:
from pathlib import Path
from bgbias.project import load_project_data
from bgbias.utils import DRIVE_ROOT

data = load_project_data(REPO_DIR)   # splits / manifests from the repo or from MyDrive/bgbias/week1
RUNS = DRIVE_ROOT / "runs"           # checkpoints and per-run outputs live on Drive
RUNS.mkdir(parents=True, exist_ok=True)
{name: len(df) for name, df in data.splits.items()}

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image, ImageFilter
from torchvision.models.segmentation import DeepLabV3_ResNet50_Weights, deeplabv3_resnet50
from bgbias import analysis as A, compose
from bgbias.datasets import build_transform
from bgbias.models import load_checkpoint

device = torch.device("cuda")
models = {m: load_checkpoint(RUNS / f"{m}_seed0" / "best.pt", data.num_classes, device)[0] for m in ("resnet_A", "resnet_B")}
breed_of = dict(zip(data.splits["test"].label, data.splits["test"].breed))
to_tensor = build_transform(train=False)

seg_weights = DeepLabV3_ResNet50_Weights.COCO_WITH_VOC_LABELS_V1
seg = deeplabv3_resnet50(weights=seg_weights).to(device).eval()
seg_tf = seg_weights.transforms()
PET_CLASSES = [seg_weights.meta["categories"].index(c) for c in ("cat", "dog")]

@torch.no_grad()
def pet_alpha(img):
    labels = seg(seg_tf(img)[None].to(device))["out"][0].argmax(0).cpu().numpy()
    mask = Image.fromarray(np.isin(labels, PET_CLASSES).astype(np.uint8) * 255).resize(img.size, Image.BILINEAR)
    return np.asarray(mask.filter(ImageFilter.GaussianBlur(1.5)), dtype=np.float32) / 255

@torch.no_grad()
def top3(img):
    x = to_tensor(img)[None].to(device)
    out = {}
    for m, model in models.items():
        p = model(x).softmax(1)[0].cpu()
        out[m] = {breed_of[i].replace("_", " "): float(p[i]) for i in p.topk(3).indices.tolist()}
    return out

rng = np.random.default_rng(0)
bg_paths = [data.places_root / p for p in rng.choice(data.bg_pool("test"), 12, replace=False)]

# example photos: test images where every A seed flips on the swapped background and every B seed does not
tables = {p.name.removesuffix("_predictions.csv"): pd.read_csv(p)
          for p in (DRIVE_ROOT / "results" / "test").glob("resnet_[AB]_seed*_predictions.csv")}
idx, _ = A.flip_examples(tables, "resnet_A", "resnet_B", n=6, seed=1)
examples = [str(data.pet_root / "images" / f"{data.splits['test'].image_id.iat[i]}.jpg") for i in idx]
print(len(examples), "example photos")

In [ ]:
import gradio as gr

def run(pet, bg_index, bg_upload):
    if pet is None:
        raise gr.Error("Загрузи фото кошки или собаки")
    pet = pet.convert("RGB")
    alpha = pet_alpha(pet)
    if alpha.mean() < 0.01:
        raise gr.Error("Не нашёл на фото кошку или собаку")
    bg = bg_upload.convert("RGB") if bg_upload is not None else compose.load_image(bg_paths[bg_index or 0])
    swapped = compose.swap_background(pet, alpha, bg)
    before, after = top3(pet), top3(swapped)
    return swapped, before["resnet_A"], before["resnet_B"], after["resnet_A"], after["resnet_B"]

with gr.Blocks(title="Background bias demo") as demo:
    gr.Markdown("## Меняется ли порода, если поменять фон?\n"
                "**A** обучена на обычных фото, **B** — с заменой фона в половине случаев.")
    bg_index = gr.State(0)
    with gr.Row():
        pet = gr.Image(type="pil", label="Фото питомца")
        with gr.Column():
            gallery = gr.Gallery(value=[str(p) for p in bg_paths], label="Новый фон (кликни)", columns=4,
                                 height=260, allow_preview=False)
            bg_upload = gr.Image(type="pil", label="…или загрузи свой фон", height=160)
    def pick(evt: gr.SelectData):   # gradio passes the click only to a parameter annotated SelectData
        return evt.index
    gallery.select(pick, None, bg_index)
    go = gr.Button("Поменять фон", variant="primary")
    swapped = gr.Image(label="С новым фоном", height=320)
    with gr.Row():
        a_before = gr.Label(num_top_classes=3, label="A — оригинал")
        b_before = gr.Label(num_top_classes=3, label="B — оригинал")
        a_after = gr.Label(num_top_classes=3, label="A — новый фон")
        b_after = gr.Label(num_top_classes=3, label="B — новый фон")
    gr.Examples(examples=[[e] for e in examples], inputs=pet, label="Примеры из test (A здесь ошибалась на чужом фоне)")
    go.click(run, [pet, bg_index, bg_upload], [swapped, a_before, b_before, a_after, b_after])

demo.launch(share=True)